# 案例2：电力系统短路电流计算

## 计算原理

短路计算是电力系统设计和保护整定的重要依据。

**三相短路电流计算：**

$$I_f = \frac{V_f^{(0)}}{Z_{ff}}$$

其中 $Z_{ff}$ 为短路点的自阻抗（Zbus矩阵的对角元）。

**不对称短路的对称分量法：**

将三相不对称系统分解为三个独立的序网络：
- 正序网络：与正常运行网络相同
- 负序网络：发电机用负序电抗
- 零序网络：取决于变压器接线方式

**各序电流关系：**
- 单相接地：$I_0 = I_1 = I_2$
- 两相短路：$I_0 = 0, I_1 = -I_2$
- 两相接地：$I_0 + I_1 + I_2 = 0$（a相电流为零）

**参考教材：** 李光琦《电力系统暂态分析》第二章

## 案例模型：三节点短路测试系统

**系统结构：**
- 节点1：发电机G1（平衡节点），Xd"=0.2
- 节点2：发电机G2，Xd"=0.25
- 节点3：负荷节点（短路点）

**线路参数：**
| 线路 | R | X |
|------|---|---|
| 1-3  | 0.02 | 0.10 |
| 2-3  | 0.025 | 0.12 |
| 1-2  | 0.03 | 0.15 |

**基准值：** S_base=100MVA, V_base=10.5kV

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

from psa4teaching.models import Bus, BusType, Line
from psa4teaching.models.generator import Generator
from psa4teaching.shortcircuit import (
    calculate_three_phase_fault,
    calculate_single_line_to_ground,
    calculate_line_to_line,
    calculate_double_line_to_ground,
    calculate_gb15544,
    sequence_to_phase,
    phase_to_sequence
)

In [ ]:
# 创建系统模型
buses = [
    Bus(number=1, name='G1', bus_type=BusType.SLACK, V_specified=1.05),
    Bus(number=2, name='G2', bus_type=BusType.PV, P_specified=0.3),
    Bus(number=3, name='短路点', bus_type=BusType.PQ),
]

lines = [
    Line(from_bus=1, to_bus=3, R=0.02, X=0.10, B=0.04),
    Line(from_bus=2, to_bus=3, R=0.025, X=0.12, B=0.05),
    Line(from_bus=1, to_bus=2, R=0.03, X=0.15, B=0.06),
]

generators = [
    Generator(bus=1, name='G1', Xd_doubleprime=0.2, H=6.0),
    Generator(bus=2, name='G2', Xd_doubleprime=0.25, H=5.0),
]

print('短路测试系统：')
print('发电机：')
for gen in generators:
    print(f'  {gen.name}: 节点{gen.bus}, Xd"={gen.Xd_doubleprime}')

## 1. 三相短路计算

In [ ]:
# 计算节点3的三相短路
fault_bus = 3
S_base = 100.0  # MVA
V_base = 10.5   # kV

result_3ph = calculate_three_phase_fault(
    buses, lines, [], generators,
    fault_bus=fault_bus,
    S_base=S_base, V_base=V_base
)

print(f'=== 三相短路计算结果（节点{fault_bus}）===')
print(f'短路点自阻抗 Zff = {result_3ph.Zff:.4f} p.u.')
print(f'短路电流（标幺值） = {abs(result_3ph.fault_current):.4f} p.u.')
print(f'短路电流（有名值） = {result_3ph.fault_current_ka:.2f} kA')
print(f'\n短路后各节点电压：')
for bus_num, V in result_3ph.V_pu.items():
    print(f'  节点{bus_num}: {abs(V):.4f} p.u.')

In [ ]:
# 转移阻抗分析
print('\n=== 转移阻抗分析 ===')
print('转移阻抗 Zkf 表示发电机k对短路点f的电气距离')
print('各发电机转移阻抗：')
for bus_num, Z in result_3ph.transfer_impedances.items():
    print(f'  发电机{bus_num}→短路点: Z={abs(Z):.4f} p.u.')
    # 各发电机对短路电流的贡献
    I_contrib = abs(1.05 / Z)  # 假设电势1.05
    I_base = S_base / (np.sqrt(3) * V_base)
    I_contrib_ka = I_contrib * I_base
    print(f'    对短路电流贡献: {I_contrib_ka:.2f} kA')

## 2. 不对称短路计算

In [ ]:
# 单相接地短路（a相接地）
result_slg = calculate_single_line_to_ground(
    buses, lines, [], generators, fault_bus=fault_bus
)

print('=== 单相接地短路（a相接地）===')
print(f'序电流: I0={abs(result_slg.sequence_currents[0]):.4f}, '
      f'I1={abs(result_slg.sequence_currents[1]):.4f}, '
      f'I2={abs(result_slg.sequence_currents[2]):.4f} p.u.')
print('单相接地时: I0 ≈ I1 ≈ I2（三序串联）')
print('\n三相电流：')
Ia, Ib, Ic = result_slg.fault_currents_3phase
print(f'  Ia = {abs(Ia):.4f} p.u. (故障相电流最大)')
print(f'  Ib = {abs(Ib):.4f} p.u.')
print(f'  Ic = {abs(Ic):.4f} p.u.')

In [ ]:
# 两相短路（b、c相）
result_ll = calculate_line_to_ground(
    buses, lines, [], generators, fault_bus=fault_bus
)

print('=== 两相短路（b、c相）===')
print(f'序电流: I0={abs(result_ll.sequence_currents[0]):.4f}, '
      f'I1={abs(result_ll.sequence_currents[1]):.4f}, '
      f'I2={abs(result_ll.sequence_currents[2]):.4f} p.u.')
print('两相短路时: I0=0, I1=-I2（正序与负序并联）')
print('\n三相电流：')
Ia, Ib, Ic = result_ll.fault_currents_3phase
print(f'  Ia = {abs(Ia):.4f} p.u. (非故障相电流为0)')
print(f'  Ib = {abs(Ib):.4f} p.u.')
print(f'  Ic = {abs(Ic):.4f} p.u.')

In [ ]:
# 两相接地短路（b、c相接地）
result_dlg = calculate_double_line_to_ground(
    buses, lines, [], generators, fault_bus=fault_bus
)

print('=== 两相接地短路（b、c相接地）===')
print(f'序电流: I0={abs(result_dlg.sequence_currents[0]):.4f}, '
      f'I1={abs(result_dlg.sequence_currents[1]):.4f}, '
      f'I2={abs(result_dlg.sequence_currents[2]):.4f} p.u.')
print('两相接地时: I1 = Ea/(Z1+Z2∥Z0)')
print('\n三相电流：')
Ia, Ib, Ic = result_dlg.fault_currents_3phase
print(f'  Ia = {abs(Ia):.4f} p.u.')
print(f'  Ib = {abs(Ib):.4f} p.u.')
print(f'  Ic = {abs(Ic):.4f} p.u.')

In [ ]:
# 对称分量变换验证
print('\n=== 对称分量变换验证 ===')
# 单相接地的三相电流
Ia, Ib, Ic = result_slg.fault_currents_3phase
print(f'三相电流: Ia={abs(Ia):.4f}, Ib={abs(Ib):.6f}, Ic={abs(Ic):.6f} p.u.')

# 反变换验证
I0, I1, I2 = phase_to_sequence(Ia, Ib, Ic)
Ia_back, Ib_back, Ic_back = sequence_to_phase(I0, I1, I2)
print(f'反变换后: Ia={abs(Ia_back):.4f}, Ib={abs(Ib_back):.6f}, Ic={abs(Ic_back):.6f}')
print('变换与反变换误差:', abs(Ia-Ia_back) + abs(Ib-Ib_back) + abs(Ic-Ic_back))

## 3. GB/T 15544 标准计算

In [ ]:
# GB15544标准计算
result_gb = calculate_gb15544(
    buses, lines, [], generators,
    fault_bus=fault_bus,
    V_nominal=10.5,
    max_current=True
)

print('=== GB/T 15544 短路电流计算 ===')
print(f'电压等级: {result_gb.voltage_level} (10.5kV属于中压)')
print(f'电压系数 c = {result_gb.c_used}')
print(f'等效阻抗 Z_eq = {abs(result_gb.Z_eq):.4f} p.u.')
print(f'X/R比值 = {result_gb.X_R_ratio:.2f}')
print('\n短路电流结果：')
print(f'  初始对称短路电流 I"k = {result_gb.Ik_initial:.2f} kA')
print(f'  峰值短路电流 ip = {result_gb.ip_peak:.2f} kA')
print(f'  稳态短路电流 Ik = {result_gb.Ik_steady:.2f} kA')
print(f'  最小短路电流 = {result_gb.Ik_min:.2f} kA')

In [ ]:
# 各类短路电流对比
print('\n=== 各类短路电流对比 ===')

# 计算有名值
I_base = S_base / (np.sqrt(3) * V_base)

short_circuit_types = ['三相短路', '单相接地', '两相短路', '两相接地']
currents_pu = [
    abs(result_3ph.fault_current),
    abs(result_slg.fault_current),
    max(abs(result_ll.fault_currents_3phase[1]), abs(result_ll.fault_currents_3phase[2])),
    max(abs(result_dlg.fault_currents_3phase[1]), abs(result_dlg.fault_currents_3phase[2]))
]
currents_kA = [I * I_base for I in currents_pu]

print('故障类型      短路电流(p.u.)    短路电流(kA)')
for i, (type_name, I_pu, I_kA) in enumerate(zip(short_circuit_types, currents_pu, currents_kA)):
    print(f'{type_name}      {I_pu:.4f}           {I_kA:.2f}')

In [ ]:
# 短路电流对比可视化
plt.figure(figsize=(10, 6))
x = range(len(short_circuit_types))
bars = plt.bar(x, currents_kA, color=['steelblue', 'coral', 'seagreen', 'purple'], alpha=0.7)
plt.xticks(x, short_circuit_types, fontsize=12)
plt.ylabel('短路电流 (kA)', fontsize=12)
plt.title(f'节点{fault_bus}各类短路电流对比', fontsize=14)
plt.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bar, val in zip(bars, currents_kA):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.2f}kA', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('shortcircuit_comparison.png', dpi=150)
plt.show()

In [ ]:
# 短路后电压分布
plt.figure(figsize=(10, 6))
nodes = list(result_3ph.V_before.keys())
V_before = [abs(result_3ph.V_before[n]) for n in nodes]
V_after = [abs(result_3ph.V_pu[n]) for n in nodes]

x = np.arange(len(nodes))
width = 0.35

plt.bar(x - width/2, V_before, width, label='短路前', color='steelblue', alpha=0.7)
plt.bar(x + width/2, V_after, width, label='短路后', color='coral', alpha=0.7)
plt.xticks(x, [f'节点{n}' for n in nodes], fontsize=12)
plt.ylabel('电压幅值 (p.u.)', fontsize=12)
plt.title(f'三相短路前后电压分布对比', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('shortcircuit_voltage.png', dpi=150)
plt.show()

## 结果分析

1. **短路电流大小：** 三相短路电流最大，单相接地次之，两相短路最小

2. **序网连接关系：**
   - 三相短路：仅正序网络
   - 单相接地：三序串联
   - 两相短路：正序与负序并联
   - 两相接地：正序与（负序∥零序）并联

3. **电压跌落：** 短路点电压跌落最大，邻近节点电压不同程度下降

4. **GB15544标准：** 提供了符合国标的短路电流计算方法，包含峰值电流和最小短路电流